In [22]:
import pandas as pd

df = pd.read_csv("data/merged_data.csv")
print(df.shape)
df.head()
print("Before dropping duplicates:", df.shape)
df = df.drop_duplicates(subset='ID', keep='first')
print("After dropping duplicates:", df.shape)

(36457, 19)
Before dropping duplicates: (36457, 19)
After dropping duplicates: (36457, 19)


In [23]:
# Age in years
df['AGE_YEARS'] = (-df['DAYS_BIRTH']) / 365

# Fix the employment placeholder issue
df['IS_UNEMPLOYED'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['DAYS_EMPLOYED_CLEAN'] = df['DAYS_EMPLOYED'].replace(365243, 0)
df['YEARS_EMPLOYED'] = (-df['DAYS_EMPLOYED_CLEAN']) / 365
df.loc[df['IS_UNEMPLOYED'] == 1, 'YEARS_EMPLOYED'] = 0

print("Done")

Done


In [24]:
print(df['OCCUPATION_TYPE'].isnull().sum())
print(df['OCCUPATION_TYPE'].isnull().mean() * 100)

11323
31.05850728255205


In [25]:
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Unknown')

print(df['OCCUPATION_TYPE'].isnull().sum())
print(df['OCCUPATION_TYPE'].value_counts().head(10))

0
OCCUPATION_TYPE
Unknown                  11323
Laborers                  6211
Core staff                3591
Sales staff               3485
Managers                  3012
Drivers                   2138
High skill tech staff     1383
Accountants               1241
Medicine staff            1207
Cooking staff              655
Name: count, dtype: int64


In [26]:
df['CODE_GENDER'] = df['CODE_GENDER'].map({'M': 1, 'F': 0})
df['FLAG_OWN_CAR'] = df['FLAG_OWN_CAR'].map({'Y': 1, 'N': 0})
df['FLAG_OWN_REALTY'] = df['FLAG_OWN_REALTY'].map({'Y': 1, 'N': 0})

print(df[['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY']].head())

   CODE_GENDER  FLAG_OWN_CAR  FLAG_OWN_REALTY
0            1             1                1
1            1             1                1
2            1             1                1
3            0             0                1
4            0             0                1


In [27]:
categorical_cols = ['NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 
                     'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(df.shape)
df.head()

(36457, 53)


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,...,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Unknown,OCCUPATION_TYPE_Waiters/barmen staff
0,5008804,1,1,1,0,427500.0,-12005,-4542,1,1,...,False,False,False,False,False,False,False,False,True,False
1,5008805,1,1,1,0,427500.0,-12005,-4542,1,1,...,False,False,False,False,False,False,False,False,True,False
2,5008806,1,1,1,0,112500.0,-21474,-1134,1,0,...,False,False,False,False,False,False,False,True,False,False
3,5008808,0,0,1,0,270000.0,-19110,-3051,1,0,...,False,False,False,False,False,True,False,False,False,False
4,5008809,0,0,1,0,270000.0,-19110,-3051,1,0,...,False,False,False,False,False,True,False,False,False,False


In [28]:
df.drop(columns=['ID', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_EMPLOYED_CLEAN', 'FLAG_MOBIL'], inplace=True)

print(df.shape)
df.columns.tolist()

(36457, 48)


['CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'FLAG_WORK_PHONE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'CNT_FAM_MEMBERS',
 'TARGET',
 'AGE_YEARS',
 'IS_UNEMPLOYED',
 'YEARS_EMPLOYED',
 'NAME_INCOME_TYPE_Pensioner',
 'NAME_INCOME_TYPE_State servant',
 'NAME_INCOME_TYPE_Student',
 'NAME_INCOME_TYPE_Working',
 'NAME_EDUCATION_TYPE_Higher education',
 'NAME_EDUCATION_TYPE_Incomplete higher',
 'NAME_EDUCATION_TYPE_Lower secondary',
 'NAME_EDUCATION_TYPE_Secondary / secondary special',
 'NAME_FAMILY_STATUS_Married',
 'NAME_FAMILY_STATUS_Separated',
 'NAME_FAMILY_STATUS_Single / not married',
 'NAME_FAMILY_STATUS_Widow',
 'NAME_HOUSING_TYPE_House / apartment',
 'NAME_HOUSING_TYPE_Municipal apartment',
 'NAME_HOUSING_TYPE_Office apartment',
 'NAME_HOUSING_TYPE_Rented apartment',
 'NAME_HOUSING_TYPE_With parents',
 'OCCUPATION_TYPE_Cleaning staff',
 'OCCUPATION_TYPE_Cooking staff',
 'OCCUPATION_TYPE_Core staff',
 'OCCUPATION_TYPE_Drivers',
 'OCCUPATION_TYP

In [29]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['TARGET'])
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train distribution:\n", y_train.value_counts(normalize=True))
print("y_test distribution:\n", y_test.value_counts(normalize=True))

X_train: (29165, 47)
X_test: (7292, 47)
y_train distribution:
 TARGET
0    0.88229
1    0.11771
Name: proportion, dtype: float64
y_test distribution:
 TARGET
0    0.882337
1    0.117663
Name: proportion, dtype: float64


In [30]:
X_train.to_csv("data/X_train.csv", index=False)
X_test.to_csv("data/X_test.csv", index=False)
y_train.to_csv("data/y_train.csv", index=False)
y_test.to_csv("data/y_test.csv", index=False)

print("Saved all processed files")

Saved all processed files


In [31]:
print([c for c in df.columns if c.startswith('NAME_INCOME_TYPE')])
print([c for c in df.columns if c.startswith('NAME_EDUCATION_TYPE')])
print([c for c in df.columns if c.startswith('NAME_FAMILY_STATUS')])
print([c for c in df.columns if c.startswith('NAME_HOUSING_TYPE')])
print([c for c in df.columns if c.startswith('OCCUPATION_TYPE')])

['NAME_INCOME_TYPE_Pensioner', 'NAME_INCOME_TYPE_State servant', 'NAME_INCOME_TYPE_Student', 'NAME_INCOME_TYPE_Working']
['NAME_EDUCATION_TYPE_Higher education', 'NAME_EDUCATION_TYPE_Incomplete higher', 'NAME_EDUCATION_TYPE_Lower secondary', 'NAME_EDUCATION_TYPE_Secondary / secondary special']
['NAME_FAMILY_STATUS_Married', 'NAME_FAMILY_STATUS_Separated', 'NAME_FAMILY_STATUS_Single / not married', 'NAME_FAMILY_STATUS_Widow']
['NAME_HOUSING_TYPE_House / apartment', 'NAME_HOUSING_TYPE_Municipal apartment', 'NAME_HOUSING_TYPE_Office apartment', 'NAME_HOUSING_TYPE_Rented apartment', 'NAME_HOUSING_TYPE_With parents']
['OCCUPATION_TYPE_Cleaning staff', 'OCCUPATION_TYPE_Cooking staff', 'OCCUPATION_TYPE_Core staff', 'OCCUPATION_TYPE_Drivers', 'OCCUPATION_TYPE_HR staff', 'OCCUPATION_TYPE_High skill tech staff', 'OCCUPATION_TYPE_IT staff', 'OCCUPATION_TYPE_Laborers', 'OCCUPATION_TYPE_Low-skill Laborers', 'OCCUPATION_TYPE_Managers', 'OCCUPATION_TYPE_Medicine staff', 'OCCUPATION_TYPE_Private servi